In [0]:
from pyspark.sql.functions import col

In [0]:
def rows_and_columns_count(df):
    return df.count(),len(df.columns)

In [0]:
# Function for checking whether we have duplicates in the given dataframe
def check_duplicates(df, colm):
    a = df.select(colm).distinct().count()
    b = df.select(colm).count()
    if a == b:
        print("No duplicates found")
    else:
        c = b-a
        print(f"Number of duplicates found: {c}")

In [0]:
# Function for checking missing values like Null in the given dataframe
def check_missing_values(df, list_col):
    missing_values = {}
    for i in list_col:
        a = df.filter(df[i].isNull()).count()
        missing_values[i] = a
    return missing_values

In [0]:

# Function for checking missing value percentage
def check_missing_values_percentage(df, list_col):
    missing_values_percent = {}
    b = df.count()
    for i in list_col:
        a = df.filter(col(i).isNull()).count()
        c = (a/b) * 100
        missing_values_percent[i] = c
    return missing_values_percent

In [0]:
# Function for checking missing value percentage for each column wise
def check_missing_values_percentage_v1(df, list_col):
    global missing_values_percent_less_than_75
    global missing_values_percent_more_than_75
    missing_values_percent_less_than_75 = {}
    missing_values_percent_more_than_75 = {}
    b = df.count()
    for i in list_col:
        a = df.filter(col(i).isNull()).count()
        c = (a/b) * 100
        if c >= 75:
            missing_values_percent_more_than_75[i] = c
        else:
            missing_values_percent_less_than_75[i] = c
    return ({"missing_values_percent_more_than_75: ": missing_values_percent_more_than_75, "missing_values_percent_less_than_75: ": missing_values_percent_less_than_75})


In [0]:
# Function for checking NaN values in the given dataframe especially for string columns
def check_string_value_as_nan(df):
    result = {}
    for i in df.columns:
        result[i] = df.filter(df[i].like("NaN")).count()
    return result

In [0]:
# Function to drop column
def drop_column(df, col_list):
    for i in col_list:
        df = df.drop(i)
        print("Columns Dropped: ",i)
    return df

In [0]:
# Function for loading the dataframe into Silver layer
def write_to_silver(df, file_name):
    silver_path = "abfss://denis@adlsdatadenis.dfs.core.windows.net/medallion/silver/"
    temp_path = f"{silver_path}/output_temp"
    final_path = f"{silver_path}/{file_name}"
    df.write.mode("overwrite").option("header","true").csv(temp_path)
    files = dbutils.fs.ls(temp_path)
    csv_file = [file.path for file in files if file.path.endswith(".csv")][0]
    print(csv_file)
    dbutils.fs.mv(csv_file,final_path)
    dbutils.fs.rm(temp_path,recurse=True)
    return f"File - {file_name} written to Silver layer successfully"

In [0]:
# Function for loading the dataframe into Silver layer
def write_to_gold(df, file_name):
    gold_path = "abfss://denis@adlsdatadenis.dfs.core.windows.net/medallion/gold/"
    temp_path = f"{gold_path}/output_temp"
    final_path = f"{gold_path}/{file_name}"
    df.write.mode("overwrite").option("header","true").csv(temp_path)
    files = dbutils.fs.ls(temp_path)
    csv_file = [file.path for file in files if file.path.endswith(".csv")][0]
    print(csv_file)
    dbutils.fs.mv(csv_file,final_path)
    dbutils.fs.rm(temp_path,recurse=True)
    return f"File - {file_name} written to Gold layer successfully"